In [ ]:
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install sentence-transformers
!pip install tqdm
!pip install faiss-cpu

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — API key + connection test
#           Tests BOTH movie and TV search endpoints
# ════════════════════════════════════════════════════════════════

import requests

TMDB_API_KEY = "62e669489429009d8a518b6366fe113b"
BASE   = "https://api.themoviedb.org/3"
PARAMS = {"api_key": TMDB_API_KEY, "language": "en-US"}

print("Testing TMDB API key for Movies...")
resp = requests.get(f"{BASE}/search/movie", params={**PARAMS, "query": "John Wick"}, timeout=10)
if resp.status_code == 200 and resp.json().get('results'):
    m = resp.json()['results'][0]
    print(f"   Movie test OK: {m['title']} ({m.get('release_date','')[:4]})")
else:
    print(f"   Movie test FAILED — status {resp.status_code}")

print("Testing TMDB API key for TV Series...")
resp2 = requests.get(f"{BASE}/search/tv", params={**PARAMS, "query": "Breaking Bad"}, timeout=10)
if resp2.status_code == 200 and resp2.json().get('results'):
    t = resp2.json()['results'][0]
    print(f"   TV test OK   : {t['name']} ({t.get('first_air_date','')[:4]})")
else:
    print(f"   TV test FAILED — status {resp2.status_code}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Build combined Movie + TV Series database from TMDB
# Fetches ~500 movies AND ~300 TV shows; saves to media_database.json
# Safe to re-run — skips entries already in the database.
# ════════════════════════════════════════════════════════════════

import requests, json, time, os
from tqdm import tqdm

BASE   = "https://api.themoviedb.org/3"
PARAMS = {"api_key": TMDB_API_KEY, "language": "en-US"}
DB_FILE = "media_database.json"   # renamed from movie_database.json


# ─────────────────────────────────────────────────────────────────────────────
# MOVIE helpers
# ─────────────────────────────────────────────────────────────────────────────

def tmdb_fetch_movie(movie_id: int) -> dict | None:
    """Fetch full metadata for a TMDB movie."""
    try:
        detail  = requests.get(f"{BASE}/movie/{movie_id}",          params=PARAMS, timeout=10).json()
        credits = requests.get(f"{BASE}/movie/{movie_id}/credits",  params=PARAMS, timeout=10).json()
        kw_resp = requests.get(f"{BASE}/movie/{movie_id}/keywords", params=PARAMS, timeout=10).json()

        title = detail.get('title', '')
        if not title:
            return None

        return {
            "tmdb_id"     : movie_id,
            "content_type": "Movie",          # ← NEW: marks it as a movie
            "title"       : title,
            "year"        : detail.get('release_date', '')[:4],
            "genres"      : [g['name'] for g in detail.get('genres', [])],
            "cast"        : [c['name'] for c in credits.get('cast', [])[:8]],
            "directors"   : [c['name'] for c in credits.get('crew', []) if c.get('job') == 'Director'],
            "creators"    : [],
            "plot"        : detail.get('overview', ''),
            "keywords"    : [k['name'] for k in kw_resp.get('keywords', [])[:10]],
            "rating"      : detail.get('vote_average', None),
            "seasons"     : None,
        }
    except Exception:
        return None


# ─────────────────────────────────────────────────────────────────────────────
# TV SERIES helpers  (NEW)
# ─────────────────────────────────────────────────────────────────────────────

def tmdb_fetch_tv(tv_id: int) -> dict | None:
    """
    Fetch full metadata for a TMDB TV series.
    TV uses different field names: 'name', 'first_air_date', 'created_by'.
    """
    try:
        detail  = requests.get(f"{BASE}/tv/{tv_id}",          params=PARAMS, timeout=10).json()
        credits = requests.get(f"{BASE}/tv/{tv_id}/credits",  params=PARAMS, timeout=10).json()
        # TV keywords endpoint returns 'results', not 'keywords'
        kw_resp = requests.get(f"{BASE}/tv/{tv_id}/keywords", params=PARAMS, timeout=10).json()

        title = detail.get('name', '')      # TV uses 'name', not 'title'
        if not title:
            return None

        # 'created_by' is a list of dicts with 'name'
        creators = [c['name'] for c in detail.get('created_by', [])]

        # TV keywords are under 'results' key
        keywords = [k['name'] for k in kw_resp.get('results', [])[:10]]

        return {
            "tmdb_id"     : tv_id,
            "content_type": "TV Series",     # ← marks it as a TV show
            "title"       : title,
            "year"        : detail.get('first_air_date', '')[:4],
            "genres"      : [g['name'] for g in detail.get('genres', [])],
            "cast"        : [c['name'] for c in credits.get('cast', [])[:8]],
            "directors"   : [],              # TV has no per-show director field
            "creators"    : creators,        # show-runners / writers
            "plot"        : detail.get('overview', ''),
            "keywords"    : keywords,
            "rating"      : detail.get('vote_average', None),
            "seasons"     : detail.get('number_of_seasons', None),
        }
    except Exception:
        return None


def content_to_text(item: dict) -> str:
    """
    Convert a movie OR TV show dict into an embedding text.
    Genres are doubled for emphasis; creators are treated like directors.
    """
    parts = [
        " ".join(item.get('genres', [])),
        " ".join(item.get('genres', [])),          # repeated for weight
        " ".join(item.get('directors', [])),
        " ".join(item.get('creators', [])),         # NEW: TV creators
        " ".join(item.get('cast', [])[:5]),
        item.get('plot', ''),
        " ".join(item.get('keywords', [])),
    ]
    return " ".join(p for p in parts if p).strip()


# ── Load or create database ───────────────────────────────────────────────────
if os.path.exists(DB_FILE):
    with open(DB_FILE, 'r') as f:
        media_db = json.load(f)
    print(f"Loaded existing DB with {len(media_db)} entries.")
else:
    # Migrate from old movie_database.json if it exists
    if os.path.exists('movie_database.json'):
        with open('movie_database.json', 'r') as f:
            old_db = json.load(f)
        # Tag all old entries as movies and add missing fields
        media_db = {}
        for k, v in old_db.items():
            v.setdefault('content_type', 'Movie')
            v.setdefault('creators', [])
            v.setdefault('seasons', None)
            media_db[k] = v
        print(f"Migrated {len(media_db)} movies from old movie_database.json.")
    else:
        media_db = {}
        print("Starting fresh database.")


def save_db():
    with open(DB_FILE, 'w') as f:
        json.dump(media_db, f, indent=2)


# ── Phase 1a: Discover MOVIE IDs ──────────────────────────────────────────────
MOVIE_SOURCES = [
    ("movie/popular",      5,  "Popular Movies"),
    ("movie/top_rated",    5,  "Top Rated Movies"),
    ("movie/now_playing",  2,  "Now Playing"),
    ("trending/movie/week",3,  "Trending Movies"),
]
MOVIE_GENRE_IDS = {
    "Action": 28, "Adventure": 12, "Animation": 16, "Comedy": 35,
    "Crime": 80, "Documentary": 99, "Drama": 18, "Fantasy": 14,
    "Horror": 27, "Mystery": 9648, "Romance": 10749,
    "Science Fiction": 878, "Thriller": 53,
}

# ── Phase 1b: Discover TV SERIES IDs  (NEW) ───────────────────────────────────
TV_SOURCES = [
    ("tv/popular",        4,  "Popular TV"),
    ("tv/top_rated",      4,  "Top Rated TV"),
    ("trending/tv/week",  3,  "Trending TV"),
]
TV_GENRE_IDS = {
    "Action & Adventure": 10759, "Animation": 16, "Comedy": 35,
    "Crime": 80, "Documentary": 99, "Drama": 18, "Family": 10751,
    "Kids": 10762, "Mystery": 9648, "Reality": 10764,
    "Sci-Fi & Fantasy": 10765, "Soap": 10766, "Talk": 10767,
    "War & Politics": 10768, "Western": 37,
}

print("\n── Phase 1a: Collecting MOVIE IDs ──")
movie_ids = set()
tasks = [(ep, pg, lb) for ep, pgs, lb in MOVIE_SOURCES for pg in range(1, pgs+1)]
for endpoint, page, label in tqdm(tasks, desc="Movie curated lists"):
    try:
        r = requests.get(f"{BASE}/{endpoint}", params={**PARAMS, "page": page}, timeout=10)
        for m in r.json().get('results', []):
            movie_ids.add(m['id'])
    except Exception:
        pass

genre_tasks = [(gn, gid, pg) for gn, gid in MOVIE_GENRE_IDS.items() for pg in range(1,4)]
for genre_name, genre_id, page in tqdm(genre_tasks, desc="Movie genre discovery"):
    try:
        r = requests.get(f"{BASE}/discover/movie",
                         params={**PARAMS, "with_genres": genre_id,
                                 "sort_by": "vote_count.desc",
                                 "vote_count.gte": 1000, "page": page}, timeout=10)
        for m in r.json().get('results', []):
            movie_ids.add(m['id'])
    except Exception:
        pass
print(f"Collected {len(movie_ids)} unique movie IDs.")


print("\n── Phase 1b: Collecting TV SERIES IDs ──")
tv_ids = set()
tv_tasks = [(ep, pg, lb) for ep, pgs, lb in TV_SOURCES for pg in range(1, pgs+1)]
for endpoint, page, label in tqdm(tv_tasks, desc="TV curated lists"):
    try:
        r = requests.get(f"{BASE}/{endpoint}", params={**PARAMS, "page": page}, timeout=10)
        for t in r.json().get('results', []):
            tv_ids.add(t['id'])
    except Exception:
        pass

tv_genre_tasks = [(gn, gid, pg) for gn, gid in TV_GENRE_IDS.items() for pg in range(1,3)]
for genre_name, genre_id, page in tqdm(tv_genre_tasks, desc="TV genre discovery"):
    try:
        r = requests.get(f"{BASE}/discover/tv",
                         params={**PARAMS, "with_genres": genre_id,
                                 "sort_by": "vote_count.desc",
                                 "vote_count.gte": 500, "page": page}, timeout=10)
        for t in r.json().get('results', []):
            tv_ids.add(t['id'])
    except Exception:
        pass
print(f"Collected {len(tv_ids)} unique TV series IDs.")


# ── Phase 2a: Fetch full movie details ───────────────────────────────────────
print("\n── Phase 2a: Fetching full MOVIE details ──")
existing_movie_tmdb_ids = {v['tmdb_id'] for v in media_db.values()
                            if v.get('content_type') == 'Movie' and 'tmdb_id' in v}
movie_ids_to_fetch = [mid for mid in movie_ids if mid not in existing_movie_tmdb_ids]
print(f"Need to fetch: {len(movie_ids_to_fetch)} (skipping {len(existing_movie_tmdb_ids)} cached)")

added_m, failed_m = 0, 0
for i, movie_id in enumerate(tqdm(movie_ids_to_fetch, desc="Fetching movies")):
    data = tmdb_fetch_movie(movie_id)
    if data and (data.get('plot') or data.get('genres')):
        key = f"movie:{data['title']}"   # prefix key to avoid collision with TV shows
        media_db[key] = data
        added_m += 1
    else:
        failed_m += 1
    if (i + 1) % 50 == 0:
        save_db()
    time.sleep(0.25)
save_db()
print(f"Movies — Added: {added_m} | Failed/empty: {failed_m}")


# ── Phase 2b: Fetch full TV series details  (NEW) ────────────────────────────
print("\n── Phase 2b: Fetching full TV SERIES details ──")
existing_tv_tmdb_ids = {v['tmdb_id'] for v in media_db.values()
                         if v.get('content_type') == 'TV Series' and 'tmdb_id' in v}
tv_ids_to_fetch = [tid for tid in tv_ids if tid not in existing_tv_tmdb_ids]
print(f"Need to fetch: {len(tv_ids_to_fetch)} (skipping {len(existing_tv_tmdb_ids)} cached)")

added_t, failed_t = 0, 0
for i, tv_id in enumerate(tqdm(tv_ids_to_fetch, desc="Fetching TV series")):
    data = tmdb_fetch_tv(tv_id)
    if data and (data.get('plot') or data.get('genres')):
        key = f"tv:{data['title']}"    # prefix to avoid key collision
        media_db[key] = data
        added_t += 1
    else:
        failed_t += 1
    if (i + 1) % 50 == 0:
        save_db()
    time.sleep(0.25)
save_db()
print(f"TV Series — Added: {added_t} | Failed/empty: {failed_t}")

total_movies = sum(1 for v in media_db.values() if v.get('content_type') == 'Movie')
total_tv     = sum(1 for v in media_db.values() if v.get('content_type') == 'TV Series')
print(f"\n Database complete!")
print(f"   Movies   : {total_movies}")
print(f"   TV Series: {total_tv}")
print(f"   Total    : {len(media_db)}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Encode ALL entries (movies + TV) into vectors
#           and build a single unified FAISS index.
# Run once after Cell 2.
# ════════════════════════════════════════════════════════════════

import faiss
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

with open('media_database.json', 'r') as f:
    media_db = json.load(f)

if len(media_db) == 0:
    raise RuntimeError(" media_database.json is empty! Run Cell 2 first.")

db_items = list(media_db.values())
total_movies = sum(1 for m in db_items if m.get('content_type') == 'Movie')
total_tv     = sum(1 for m in db_items if m.get('content_type') == 'TV Series')
print(f"Loaded {len(db_items)} entries ({total_movies} movies, {total_tv} TV series).")

print("\nLoading embedding model...")
model = SentenceTransformer('all-mpnet-base-v2')
print("Model loaded.")

# content_to_text must be defined here (same as Cell 2)
def content_to_text(item: dict) -> str:
    parts = [
        " ".join(item.get('genres', [])),
        " ".join(item.get('genres', [])),
        " ".join(item.get('directors', [])),
        " ".join(item.get('creators', [])),
        " ".join(item.get('cast', [])[:5]),
        item.get('plot', ''),
        " ".join(item.get('keywords', [])),
    ]
    return " ".join(p for p in parts if p).strip()

texts  = [content_to_text(m) for m in db_items]
titles = [m['title'] for m in db_items]

print(f"\nEncoding {len(texts)} entries into vectors...")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
if embeddings.ndim != 2 or embeddings.shape[0] == 0:
    raise RuntimeError(f" Bad embeddings shape: {embeddings.shape}")

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype('float32'))

faiss.write_index(index, 'media.faiss')        # renamed from movies.faiss
np.save('embeddings.npy', embeddings)
with open('media_titles.json', 'w') as f:       # renamed from movie_titles.json
    json.dump(titles, f)

print(f"\n FAISS index built: {index.ntotal} vectors, dim={dim}")
print("Saved: media.faiss | embeddings.npy | media_titles.json")

In [1]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Load everything and run recommendations
#           Works for BOTH movies and TV series.
# Run this cell every time you want recommendations.
# ════════════════════════════════════════════════════════════════

import faiss, json, requests, numpy as np
from sentence_transformers import SentenceTransformer

print("Loading model and index...")
model = SentenceTransformer('all-mpnet-base-v2')
index = faiss.read_index('media.faiss')

with open('media_database.json', 'r') as f:
    media_db = json.load(f)
with open('media_titles.json', 'r') as f:
    db_titles = json.load(f)

db_items = list(media_db.values())
total_movies = sum(1 for m in db_items if m.get('content_type') == 'Movie')
total_tv     = sum(1 for m in db_items if m.get('content_type') == 'TV Series')
print(f" Ready — {len(db_items)} entries indexed ({total_movies} movies + {total_tv} TV series).")

TMDB_API_KEY = "62e669489429009d8a518b6366fe113b"
BASE   = "https://api.themoviedb.org/3"
PARAMS = {"api_key": TMDB_API_KEY, "language": "en-US"}


# ─────────────────────────────────────────────────────────────────────────────
# Fetch helpers (same as Cell 2, needed for live TMDB lookups of user input)
# ─────────────────────────────────────────────────────────────────────────────

def tmdb_fetch_movie(movie_id: int) -> dict | None:
    try:
        detail  = requests.get(f"{BASE}/movie/{movie_id}",          params=PARAMS, timeout=10).json()
        credits = requests.get(f"{BASE}/movie/{movie_id}/credits",  params=PARAMS, timeout=10).json()
        kw_resp = requests.get(f"{BASE}/movie/{movie_id}/keywords", params=PARAMS, timeout=10).json()
        title = detail.get('title', '')
        if not title: return None
        return {
            "tmdb_id": movie_id, "content_type": "Movie", "title": title,
            "year": detail.get('release_date', '')[:4],
            "genres": [g['name'] for g in detail.get('genres', [])],
            "cast": [c['name'] for c in credits.get('cast', [])[:8]],
            "directors": [c['name'] for c in credits.get('crew', []) if c.get('job') == 'Director'],
            "creators": [],
            "plot": detail.get('overview', ''),
            "keywords": [k['name'] for k in kw_resp.get('keywords', [])[:10]],
            "rating": detail.get('vote_average', None),
            "seasons": None,
        }
    except Exception: return None


def tmdb_fetch_tv(tv_id: int) -> dict | None:
    try:
        detail  = requests.get(f"{BASE}/tv/{tv_id}",          params=PARAMS, timeout=10).json()
        credits = requests.get(f"{BASE}/tv/{tv_id}/credits",  params=PARAMS, timeout=10).json()
        kw_resp = requests.get(f"{BASE}/tv/{tv_id}/keywords", params=PARAMS, timeout=10).json()
        title = detail.get('name', '')
        if not title: return None
        return {
            "tmdb_id": tv_id, "content_type": "TV Series", "title": title,
            "year": detail.get('first_air_date', '')[:4],
            "genres": [g['name'] for g in detail.get('genres', [])],
            "cast": [c['name'] for c in credits.get('cast', [])[:8]],
            "directors": [],
            "creators": [c['name'] for c in detail.get('created_by', [])],
            "plot": detail.get('overview', ''),
            "keywords": [k['name'] for k in kw_resp.get('results', [])[:10]],
            "rating": detail.get('vote_average', None),
            "seasons": detail.get('number_of_seasons', None),
        }
    except Exception: return None


def tmdb_search_any(query: str) -> dict | None:
    """
    Smart search: tries movie first, then TV series.
    Returns whichever result has the higher vote_count (more popular match).
    Falls back to whichever returned any result at all.
    """
    movie_result, tv_result = None, None

    # Search movies
    try:
        r = requests.get(f"{BASE}/search/movie", params={**PARAMS, "query": query}, timeout=10)
        hits = r.json().get('results', [])
        if hits:
            movie_result = (hits[0].get('vote_count', 0), hits[0]['id'], 'movie')
    except Exception: pass

    # Search TV series
    try:
        r = requests.get(f"{BASE}/search/tv", params={**PARAMS, "query": query}, timeout=10)
        hits = r.json().get('results', [])
        if hits:
            tv_result = (hits[0].get('vote_count', 0), hits[0]['id'], 'tv')
    except Exception: pass

    # Pick the result with more votes (closer match to what user intended)
    candidates = [c for c in [movie_result, tv_result] if c is not None]
    if not candidates:
        print(f" No results found for '{query}'")
        return None

    best = max(candidates, key=lambda x: x[0])
    _, best_id, media_type = best

    if media_type == 'movie':
        data = tmdb_fetch_movie(best_id)
    else:
        data = tmdb_fetch_tv(best_id)

    if data:
        ctype = data['content_type']
        seasons_str = f" | {data['seasons']} seasons" if data.get('seasons') else ""
        print(f" Found [{ctype}]: {data['title']} ({data['year']}) | {', '.join(data['genres'][:3])}{seasons_str}")
    return data


def content_to_text(item: dict) -> str:
    parts = [
        " ".join(item.get('genres', [])),
        " ".join(item.get('genres', [])),
        " ".join(item.get('directors', [])),
        " ".join(item.get('creators', [])),
        " ".join(item.get('cast', [])[:5]),
        item.get('plot', ''),
        " ".join(item.get('keywords', [])),
    ]
    return " ".join(p for p in parts if p).strip()


# ─────────────────────────────────────────────────────────────────────────────
# Explanation engine
# ─────────────────────────────────────────────────────────────────────────────

def explain_match(input_items: list, recommended: dict) -> list:
    """Return human-readable reasons why an item was recommended."""
    rec_genres   = set(recommended.get('genres', []))
    rec_creators = set(recommended.get('directors', []) + recommended.get('creators', []))
    rec_cast     = set(recommended.get('cast', []))
    rec_keywords = set(recommended.get('keywords', []))

    all_genres, all_creators, all_cast, all_keywords = set(), set(), set(), set()
    for m in input_items:
        all_genres   |= set(m.get('genres', []))
        all_creators |= set(m.get('directors', []) + m.get('creators', []))
        all_cast     |= set(m.get('cast', []))
        all_keywords |= set(m.get('keywords', []))

    reasons = []
    shared_genres = rec_genres & all_genres
    if shared_genres:
        reasons.append(f" Genre match: {', '.join(sorted(shared_genres))}")
    shared_creators = rec_creators & all_creators
    if shared_creators:
        reasons.append(f" Same creator/director: {', '.join(sorted(shared_creators))}")
    shared_cast = rec_cast & all_cast
    if shared_cast:
        reasons.append(f" Shared actor(s): {', '.join(sorted(shared_cast))}")
    shared_kw = rec_keywords & all_keywords
    if shared_kw:
        reasons.append(f" Similar themes: {', '.join(sorted(shared_kw)[:4])}")
    if not reasons:
        reasons.append(" Similar mood and storytelling style (semantic similarity)")
    return reasons


# ─────────────────────────────────────────────────────────────────────────────
# Main recommendation function
# ─────────────────────────────────────────────────────────────────────────────

def recommend(titles_input: list, top_k: int = 10) -> None:
    """
    Given 1–8 movie OR TV series titles you've watched,
    find the most similar content from the entire database
    (movies + TV series combined).

    Args:
        titles_input : list of 1–8 title strings (movie or TV show names)
        top_k        : number of recommendations to return (default 10)
    """
    if not (1 <= len(titles_input) <= 8):
        print(" Please provide between 1 and 8 titles.")
        return

    print(f"\n Fetching data for {len(titles_input)} title(s)...")

    # Fetch each input title (check local DB first, fall back to live TMDB)
    input_items = []
    for title in titles_input:
        # Check both movie: and tv: prefixed keys in local DB
        cached = media_db.get(f"movie:{title}") or media_db.get(f"tv:{title}")
        if cached:
            ctype = cached.get('content_type', 'Movie')
            print(f" {title} [{ctype}] (from cache)")
            input_items.append(cached)
        else:
            data = tmdb_search_any(title)    # smart search: tries movie + TV
            if data:
                input_items.append(data)
            else:
                print(f" Could not find '{title}' — skipping.")

    if not input_items:
        print(" No valid titles found. Check your input and API key.")
        return

    # Build averaged taste vector
    input_texts   = [content_to_text(m) for m in input_items]
    input_vectors = model.encode(input_texts, normalize_embeddings=True, convert_to_numpy=True)
    taste_vector  = np.mean(input_vectors, axis=0).astype('float32').reshape(1, -1)
    norm = np.linalg.norm(taste_vector)
    if norm > 0:
        taste_vector /= norm

    # FAISS search
    scores, indices = index.search(taste_vector, top_k + len(input_items) + 5)

    input_titles_lower = {t.lower() for t in titles_input}
    shown = 0

    print(f"\n{'═'*65}")
    print(f"  TOP {top_k} RECOMMENDATIONS FOR YOU  (Movies & TV Series)")
    print(f"{'═'*65}")

    for score, idx in zip(scores[0], indices[0]):
        if shown >= top_k:
            break
        if idx < 0 or idx >= len(db_items):
            continue

        rec = db_items[idx]
        if rec['title'].lower() in input_titles_lower:
            continue

        shown += 1
        ctype   = rec.get('content_type', 'Movie')     # "Movie" or "TV Series"
        genres  = ', '.join(rec.get('genres', [])[:3])
        rating  = rec.get('rating', '?')
        year    = rec.get('year', '?')
        seasons = rec.get('seasons')
        seasons_str = f" | {seasons} seasons" if seasons else ""
        reasons = explain_match(input_items, rec)

        # Print with content type badge
        badge = "📺" if ctype == "TV Series" else "🎬"
        print(f"\n{shown}. {badge} [{ctype}] {rec['title']} ({year}){seasons_str}")
        print(f"    ⭐{rating}/10  |  {genres}")
        print(f"    Similarity: {score:.4f}")
        if rec.get('plot'):
            print(f"   {rec['plot'][:110]}...")
        print("   Why recommended:")
        for r in reasons:
            print(f"     {r}")

    if shown == 0:
        print("No recommendations found. Try different titles.")

    print(f"\n{'═'*65}")

print("\n recommend() is ready. Works for movies AND TV series!")

Loading model and index...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Ready — 1048 entries indexed (623 movies + 425 TV series).

 recommend() is ready. Works for movies AND TV series!


In [2]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Enter up to 8 movie / TV show titles you've enjoyed.
#           Mix and match freely — the system handles both!
# ════════════════════════════════════════════════════════════════

print("Enter movie OR TV show titles (one per line).")
print("Type 'OK' when finished (max 8):\n")

MY_TITLES = []
while len(MY_TITLES) < 8:
    title = input().strip()
    if title.upper() == "OK":
        break
    if title:
        MY_TITLES.append(title)

if MY_TITLES:
    print(f"\nFinding recommendations based on {len(MY_TITLES)} title(s)...\n")
    recommend(MY_TITLES, top_k=10)
else:
    print("No titles entered. Add them to MY_TITLES list and run again.")

Enter movie OR TV show titles (one per line).
Type 'OK' when finished (max 8):


Finding recommendations based on 4 title(s)...


 Fetching data for 4 title(s)...
 Found [TV Series]: SAS Rogue Heroes (2022) | Drama, War & Politics, Action & Adventure | 2 seasons
 Found [Movie]: The Ministry of Ungentlemanly Warfare (2024) | Action, Comedy, War
 Found [Movie]: War Machine (2026) | Action, Science Fiction, Thriller
 Found [TV Series]: SAS Australia (2020) | Reality | 5 seasons

═════════════════════════════════════════════════════════════════
  TOP 10 RECOMMENDATIONS FOR YOU  (Movies & TV Series)
═════════════════════════════════════════════════════════════════

1. 📺 [TV Series] SEAL Team (2017) | 7 seasons
    ⭐8.0/10  |  Action & Adventure, Drama, War & Politics
    Similarity: 0.7609
   Military drama following the professional and personal lives of the most elite unit of Navy SEALs as they trai...
   Why recommended:
      Genre match: Action & Adventure, Drama, War & Politics

2. 📺 